# SEC Filing Intelligence — notebook walkthrough

**Purpose:** understand evidence retrieval, document identity, source citations, and abstention.

**Important real-data status:** the checked-in fixture filings describe fictional companies and are **synthetic demonstration data**, not actual SEC company filings. This notebook does **not** claim real-data experimental results. To evaluate on real data, first download and vet genuine SEC EDGAR filings using the documented acquisition workflow and current SEC access rules.

This notebook is an explanatory companion created after the modular implementation, not a reconstruction of its historical development. Outputs are intentionally cleared.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "README.md").exists() and (ROOT.parent / "README.md").exists():
    ROOT = ROOT.parent
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run the notebook from the project root or notebooks/ directory")
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

## 1. Read the retrieval pipeline
The corpus is validated, split into paragraphs, indexed, retrieved, then used for *extractive* citation-checked answers.

In [ ]:
from sec_filing_intelligence.corpus import load_corpus
import inspect
print(inspect.signature(load_corpus))
print(load_corpus.__doc__ or "See src/sec_filing_intelligence/corpus.py")

## 2. Inspect sample fixture data — demonstration ONLY
The records below are fictional and must not be described as financial evidence.

In [ ]:
import json
fixture = ROOT / "data" / "fixtures" / "filings.json"
rows = json.loads(fixture.read_text(encoding="utf-8"))
print("Demo records:",len(rows))
for row in rows[:3]:
    print("Fields:",sorted(row.keys()))
    print("Document ID:",row.get("document_id",row.get("id","unknown")))
    print("Source:",row.get("source_url",row.get("url","unknown")))
    print()

## 3. Run retrieval and evaluation through the implementation
No separate notebook-specific retrieval algorithm is added: the Python source is the single source of truth. These commands operate on **synthetic fixtures** and do not establish accuracy on real filings.

In [ ]:
import subprocess
env = __import__("os").environ.copy()
env["PYTHONPATH"] = str(ROOT / "src")
cmd = [sys.executable, "-m", "sec_filing_intelligence.evaluate", "--k", "3"]
result = subprocess.run(cmd,cwd=ROOT,env=env,text=True,capture_output=True)
print("Exit code:",result.returncode)
print(result.stdout[:5000])
if result.returncode: print(result.stderr[:2000])

## 4. Explore retrieval scores and failure modes
Inspect the evaluation output and the adversarial fixture list. Document identity and abstention matter at least as much as rank. This is a teaching demonstration; real-data scores remain unverified.

In [ ]:
adversarial = json.loads((ROOT/"data"/"fixtures"/"adversarial.json").read_text(encoding="utf-8"))
print("Adversarial fixture type:",type(adversarial).__name__,"count:",len(adversarial))
print("Review sources: src/sec_filing_intelligence/retrieval.py, answer.py, evaluate.py")

## 5. Transition to reusable software
- `corpus.py`: validation, normalization, paragraph boundaries.
- `retrieval.py`: overlap and BM25/bigram rank fusion.
- `answer.py`: extractive answers, citation verification, abstention.
- `evaluate.py`: repeatable retrieval evaluation.
- `acquire.py`: opt-in SEC acquisition with provenance controls.
- `tests/`: contracts and adversarial tests.

**Next real-data milestone:** independently obtain authentic 10-K/10-Q filings, confirm rights and access requirements, and build document-grounded query judgments before reporting real-data benchmark numbers.